# `results_analyzer.py` — Experiment Results Report Generator

## Purpose
Loads `results/experiments/all_results.json` (produced by `experiment_runner.py`) and generates:
- **Markdown report** with baseline + ablation comparison tables
- **LaTeX tables** for embedding in the FYP report
- **Bar charts** comparing Macro-F1 across experiments

## Usage
```bash
python src/experiments/results_analyzer.py \
    --results results/experiments/all_results.json \
    --save_dir results/experiments/analysis
```

## Output Files
| File | Description |
|------|-------------|
| `experiment_report.md` | Full Markdown report (baseline + all ablations) |
| `baselines_macro_f1.png` | Bar chart for baseline comparisons |
| `baselines_latex.tex` | LaTeX table for B-series |
| `A1_macro_f1.png` ... | A-series bar charts |
| `ablations_latex.tex` | LaTeX table for all ablations |

In [ ]:
import argparse, json, os, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — renders to file without a display
import matplotlib.pyplot as plt
from pathlib import Path

# The 7 ABSA aspects and 3 sentiment classes used throughout the project
ASPECTS = ['stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']
CLASSES = ['negative', 'neutral', 'positive']

## Formatting Helpers

In [ ]:
def load_results(path):
    with open(path) as f: return json.load(f)

def _f(val, decimals=4):
    """Format a float to N decimal places, or return '—' if None."""
    if val is None: return '—'
    try: return f'{float(val):.{decimals}f}'
    except: return '—'

def _pct(val):
    """Format a decimal fraction as percentage string (e.g. 0.85 → '85.0%')."""
    if val is None: return '—'
    try: return f'{float(val)*100:.1f}%'
    except: return '—'

## Table Generation Functions
Each function takes the `results` dict and a list of experiment IDs, and returns a formatted Markdown string.

In [ ]:
def overall_comparison_table(results, exp_ids, title):
    """Markdown table: Accuracy, Macro-F1, Weighted-F1, MCC, Time per experiment."""
    rows = []
    for exp_id in exp_ids:
        if exp_id not in results: continue
        r = results[exp_id]
        if r['status'] != 'done' or not r.get('overall'):
            rows.append((exp_id, r.get('description', ''), '—', '—', '—', '—', '—'))
            continue
        o = r['overall']
        # Extract each overall metric; fall back to '—' if missing
        rows.append((
            exp_id, r.get('description', exp_id),
            _f(o.get('accuracy')), _f(o.get('macro_f1')), _f(o.get('weighted_f1')),
            _f(o.get('mcc')),
            f"{r.get('duration_mins', '')} min" if r.get('duration_mins') else '—',
        ))
    lines = [f'## {title}\n',
             '| Experiment | Accuracy | Macro-F1 | Weighted-F1 | MCC | Time |',
             '|-----------|----------|----------|-------------|-----|------|']
    for _, desc, acc, mf1, wf1, mcc, t in rows:
        lines.append(f'| {desc} | {acc} | **{mf1}** | {wf1} | {mcc} | {t} |')
    return '\n'.join(lines)


def per_aspect_table(results, exp_ids, title):
    """Per-aspect Macro-F1 comparison table."""
    lines = [f'## {title} — Per-Aspect Macro-F1\n',
             '| Experiment | ' + ' | '.join(ASPECTS) + ' | Avg |',
             '|---|' + '---|' * (len(ASPECTS) + 1)]
    for exp_id in exp_ids:
        if exp_id not in results: continue
        r          = results[exp_id]
        per_aspect = r.get('per_aspect', {})
        f1_vals    = []
        cells = []
        for asp in ASPECTS:
            f1 = per_aspect.get(asp, {}).get('macro_f1')
            cells.append(_f(f1))
            if f1 is not None: f1_vals.append(float(f1))
        avg = _f(np.mean(f1_vals)) if f1_vals else '—'
        lines.append(f"| {r.get('description', exp_id)} | " + ' | '.join(cells) + f' | {avg} |')
    return '\n'.join(lines)


def rare_class_table(results, exp_ids, title):
    """Per-class F1 for the rarest aspect-class combinations."""
    # These are the classes with fewest training samples — hardest to learn
    rare_cols = [('price', 'negative', 0), ('price', 'neutral', 1),
                 ('packing', 'neutral', 1), ('smell', 'neutral', 1)]
    col_headers = ' | '.join([f'{asp}-{lbl}' for asp, lbl, _ in rare_cols])
    lines = [f'## {title} — Rare Class F1\n',
             f'| Experiment | {col_headers} |',
             '|---|' + '---|' * len(rare_cols)]
    for exp_id in exp_ids:
        if exp_id not in results: continue
        r          = results[exp_id]
        per_aspect = r.get('per_aspect', {})
        cells = []
        for asp, lbl, idx in rare_cols:
            pcf1 = per_aspect.get(asp, {}).get('per_class_f1', [])
            cells.append(_f(pcf1[idx]) if idx < len(pcf1) else '—')
        lines.append(f"| {r.get('description', exp_id)} | " + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

## Chart Generation

In [ ]:
def plot_macro_f1_comparison(results, exp_ids, title, save_path):
    """Bar chart of Macro-F1 across experiments. Best model gets a distinct colour."""
    labels, values = [], []
    for exp_id in exp_ids:
        if exp_id not in results or results[exp_id]['status'] != 'done': continue
        f1 = results[exp_id].get('overall', {}).get('macro_f1')
        if f1 is not None:
            labels.append(results[exp_id].get('description', exp_id)[:30])
            values.append(float(f1))

    if not values:
        print(f'  No data for chart: {title}'); return

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.2), 5))
    # Highlight the best model in green; others in blue
    colors = ['#4a90d9' if v < max(values) else '#2ecc71' for v in values]
    bars   = ax.bar(range(len(labels)), values, color=colors, alpha=0.85,
                    edgecolor='white', linewidth=0.5)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Macro-F1')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim([0, min(1.0, max(values) + 0.1)])
    ax.yaxis.grid(True, alpha=0.3, linestyle='--'); ax.set_axisbelow(True)

    for bar, val in zip(bars, values):  # Annotate each bar with its F1 value
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Chart saved: {save_path}')

## Main Report Generator

In [ ]:
def generate_report(results, save_dir):
    save_dir.mkdir(parents=True, exist_ok=True)

    # Group experiments by prefix (B = baselines, A = ablations)
    baseline_ids = [k for k in results if k.startswith('B')]
    ablation_ids = [k for k in results if k.startswith('A')]

    # Group ablations by study (A1, A2, ... A7)
    ablation_groups = {
        f'A{i}': ([k for k in ablation_ids if k.startswith(f'A{i}')],
                  f'Ablation {i}: GCN / Aspect Attn / Loss / Aug / Head / MSR / Weights'.split(' / ')[i-1])
        for i in range(1, 8)
    }

    report_lines = ['# Experiment Results Report', f'\nGenerated from {len(results)} experiments\n', '---']

    if baseline_ids:
        report_lines.extend(['\n# Baseline Comparisons\n',
                             overall_comparison_table(results, baseline_ids, 'Overall Metrics'), '\n',
                             per_aspect_table(results, baseline_ids, 'Baseline Comparison'), '\n',
                             rare_class_table(results, baseline_ids, 'Baseline Comparison')])
        plot_macro_f1_comparison(results, baseline_ids, 'Baseline Comparison — Macro-F1',
                                 str(save_dir / 'baselines_macro_f1.png'))

    report_lines.append('\n---\n# Ablation Studies\n')
    for grp_key, (ids, grp_title) in ablation_groups.items():
        if not ids: continue
        report_lines.extend([f'\n### {grp_title}\n',
                             overall_comparison_table(results, ids, grp_title), '\n',
                             rare_class_table(results, ids, grp_title)])
        plot_macro_f1_comparison(results, ids, f'{grp_title} — Macro-F1',
                                 str(save_dir / f'{grp_key}_macro_f1.png'))

    report_path = save_dir / 'experiment_report.md'
    report_path.write_text('\n'.join(report_lines), encoding='utf-8')
    print(f'\nFull report → {report_path}')


# ── CLI entry point ────────────────────────────────────────────────────────────
if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--results',  default='results/experiments/all_results.json')
    parser.add_argument('--save_dir', default='results/experiments/analysis')
    args     = parser.parse_args()
    results  = load_results(args.results)
    generate_report(results, Path(args.save_dir))